In [1]:
import pandas as pd
import json
import ir_datasets
from src.data import DATA_DIR_PROCESSED, DATA_DIR_RAW
import os
from topic_gen.evaluate import QrelsEvaluator, CohenKappa, MeanAverageError, AreaUnderReceiver, ROSKendallTau, ROSRBO, binarize_qrels, load_runs_from_path
import ir_measures
from src.data import parse_file_names
from pathlib import Path

from topic_gen import logger
logger.setLevel("DEBUG")

In [2]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_PROCESSED / "qrels"

predictions = []
names = []
metadata_records = []
for result in os.listdir(BASE_DIR):
    names.append(result)

    # metadata
    with open(os.path.join(BASE_DIR, result, "metadata.json")) as f:
        metadata = json.load(f)
    metadata_records.append(metadata)

    # predictions
    qrels = binarize_qrels(ir_measures.read_trec_qrels(
        os.path.join(BASE_DIR, result, "qrels.csv.gz")))
    predictions.append(qrels)

In [3]:
# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "disks45/nocr/trec-robust-2004").qrels_iter()),
    measures=[CohenKappa(), MeanAverageError(), AreaUnderReceiver()],
    bootstrap=20,
    names=names)

[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 12 / 2939
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 15 / 2936
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 8 / 2943
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 9 / 2942
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 6 / 2945
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] 

In [20]:
# metadata table
metadata = pd.DataFrame(metadata_records)
metadata = metadata.join(pd.json_normalize(
    metadata["topics"]).add_prefix("topics_"))
metadata.drop(columns=["topics"], inplace=True)
metadata["topics_prompt"] = metadata["topics_prompt"].apply(lambda p: str(Path(p).stem) if pd.notnull(p) else "human")
metadata["prompt"] = metadata["prompt"].apply(lambda p: str(Path(p).stem))
metadata["model"] = metadata["model"].str.replace("-MT100", "").replace("-MT1000", "")

In [21]:
def format_score(row):
    return f"{row['value']:.2f} ± {row['ci']:.2f}"


table = pd.DataFrame(res)
table["score"] = table.apply(format_score, axis=1)
table = table.pivot(index="name", columns="measure",
                    values="score").reset_index()

In [22]:
table = table.merge(metadata, left_on="name", right_on="date")

## Prerequisite
### Can we reproduce the results from Thomas et al. with open models?
Yes! Open models outperform the results on GPT 4.1

In [23]:
table[(table["prompt"] == "-DNA-zero-shot")
      & (table["topics_prompt"] == "human")][["model", "prompt", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
0,qwen3-30B-no-think,-DNA-zero-shot,0.75 ± 0.02,0.11 ± 0.01,0.89 ± 0.01
3,qwen3-14B-no-think,-DNA-zero-shot,0.72 ± 0.03,0.13 ± 0.01,0.85 ± 0.01
46,gpt-4.1,-DNA-zero-shot,0.64 ± 0.03,0.17 ± 0.01,0.81 ± 0.01


### Do LLM judges benefit from additional information in the topic?
Binary Setting:
- Für `GPT` scheint mehr Kontext schlechter zu sein. Für `Qwen` scheint mehr Kontext besser zu sein (mit ausnahmen...).
- Genereller Effekt ist gering

In [25]:
partial_annotation_prompts = [
    "-DNA-zero-shot", "-DNA-zero-shot-masked-title", "-DNA-zero-shot-masked-description", "-DNA-zero-shot-masked-narrative", "-DNA-zero-shot-masked-title-description", "-DNA-zero-shot-masked-title-narrative", "-DNA-zero-shot-masked-description-narrative"]

table["prompt"] = pd.Categorical(table["prompt"], partial_annotation_prompts)
table[table["topics_prompt"] == "human"].sort_values(by=["model", "prompt"])[["model", "prompt", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
46,gpt-4.1,-DNA-zero-shot,0.64 ± 0.03,0.17 ± 0.01,0.81 ± 0.01
3,qwen3-14B-no-think,-DNA-zero-shot,0.72 ± 0.03,0.13 ± 0.01,0.85 ± 0.01
52,qwen3-14B-no-think,-DNA-zero-shot-masked-title,0.70 ± 0.03,0.14 ± 0.01,0.84 ± 0.01
48,qwen3-14B-no-think,-DNA-zero-shot-masked-description,0.71 ± 0.02,0.13 ± 0.01,0.85 ± 0.01
49,qwen3-14B-no-think,-DNA-zero-shot-masked-narrative,0.71 ± 0.02,0.13 ± 0.01,0.85 ± 0.01
50,qwen3-14B-no-think,-DNA-zero-shot-masked-title-description,0.63 ± 0.03,0.17 ± 0.01,0.81 ± 0.01
51,qwen3-14B-no-think,-DNA-zero-shot-masked-title-narrative,0.71 ± 0.02,0.13 ± 0.01,0.85 ± 0.01
47,qwen3-14B-no-think,-DNA-zero-shot-masked-description-narrative,0.65 ± 0.03,0.15 ± 0.01,0.86 ± 0.01
0,qwen3-30B-no-think,-DNA-zero-shot,0.75 ± 0.02,0.11 ± 0.01,0.89 ± 0.01


## Alignment
RQ: How well aligne qrels based on generated topics with qrels based on the original qrels?

### Prompting Strategies
==Important: the columns ndocpos and ndocsneg state values even if they are ignored by the prompts.==

Prompts:
- Trec: query variants and relevant docs
- trec-query: only query
- trec-contrastive: query, relevant and irelevant documents


### Findings
- Judgments based on the original topics are allways better
- Some settings show a substantial drop in alignment, for example, for `qwen3-30B` on generated topics with the `trec` prompt with one query variant and one relevant document the Cohen's $\kappa$ agreement to human labels drops from the substantial agreement 0.75 to a moderate agreement of 0.52.
- More context allways helps
- Contrastive prompting is the best

In [26]:
prompt_sorter = ["human", "trec", "trec-query", "trec-contrastive"]
table["topics_prompt"] = pd.Categorical(
    table["topics_prompt"], prompt_sorter)

table[(table["prompt"] =="-DNA-zero-shot") & ((table["model"] == table["topics_model"]) | (table["topics_model"] == "trec assessors"))]\
      .sort_values(by=["model", "topics_prompt"], ascending=[True, True])\
            [["topics_prompt", "model", "topics_nqueries", "topics_ndocspos", "topics_ndocsneg", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,topics_prompt,model,topics_nqueries,topics_ndocspos,topics_ndocsneg,CohenKappa,MeanAverageError,AreaUnderReceiver
46,human,gpt-4.1,NaN,NaN,NaN,0.64 ± 0.03,0.17 ± 0.01,0.81 ± 0.01
3,human,qwen3-14B-no-think,NaN,NaN,NaN,0.72 ± 0.03,0.13 ± 0.01,0.85 ± 0.01
26,trec,qwen3-14B-no-think,1.0,1.0,3.0,0.49 ± 0.02,0.26 ± 0.02,0.76 ± 0.01
27,trec,qwen3-14B-no-think,3.0,2.0,3.0,0.63 ± 0.03,0.18 ± 0.01,0.81 ± 0.01
28,trec,qwen3-14B-no-think,5.0,3.0,3.0,0.69 ± 0.02,0.14 ± 0.01,0.84 ± 0.01
29,trec-query,qwen3-14B-no-think,1.0,3.0,3.0,0.50 ± 0.03,0.25 ± 0.01,0.76 ± 0.01
31,trec-query,qwen3-14B-no-think,3.0,3.0,3.0,0.52 ± 0.03,0.24 ± 0.01,0.76 ± 0.01
32,trec-query,qwen3-14B-no-think,5.0,3.0,3.0,0.58 ± 0.03,0.20 ± 0.01,0.79 ± 0.01
34,trec-contrastive,qwen3-14B-no-think,1.0,1.0,1.0,0.52 ± 0.02,0.24 ± 0.01,0.77 ± 0.01
36,trec-contrastive,qwen3-14B-no-think,3.0,2.0,2.0,0.63 ± 0.03,0.18 ± 0.01,0.80 ± 0.01


## Clarity
RQ: How well agree different annotators on the relevance lable using the generated toipics?  

In [47]:
df = table[
    (table["prompt"] =="-DNA-zero-shot") & 
    (table["topics_prompt"] == "trec-contrastive") &
    (table["topics_nqueries"] == 5) &
    (table["topics_ndocsneg"] == 3) &
    (table["topics_ndocspos"] == 3) 
].copy()

df[["name", "CohenKappa", "model", "topics_model"]].pivot(
    index="model",
    columns="topics_model",
    values="CohenKappa")

topics_model,gpt-oss-120B,qwen3-14B-no-think,qwen3-30B-no-think
model,,,
qwen3-14B-no-think,0.63 ± 0.03,0.70 ± 0.02,0.62 ± 0.02
qwen3-30B-no-think,NaN,0.74 ± 0.02,0.69 ± 0.02


In [50]:
table[table["model"]=="qwen3-30B-no-think" ]

,name,AreaUnderReceiver,CohenKappa,MeanAverageError,date,model,data,prompt,k,s,...,topics_data,topics_k,topics_prompt,topics_nqueries,topics_ndocspos,topics_ndocsneg,topics_output,topics_task,topics_date,topics_s
0,2025-11-12_22:35:46,0.89 ± 0.01,0.75 ± 0.02,0.11 ± 0.01,2025-11-12_22:35:46,qwen3-30B-no-think,robust,-DNA-zero-shot,None,True,...,robust,None,human,NaN,NaN,NaN,NaN,topics,NaN,True
1,2025-11-12_22:41:45,0.76 ± 0.01,0.52 ± 0.03,0.24 ± 0.01,2025-11-12_22:41:45,qwen3-30B-no-think,robust,-DNA-zero-shot,None,True,...,robust,None,trec,1.0,1.0,3.0,../data/processed/topics,topics,NaN,NaN
2,2025-11-12_22:47:44,0.81 ± 0.01,0.64 ± 0.02,0.17 ± 0.01,2025-11-12_22:47:44,qwen3-30B-no-think,robust,-DNA-zero-shot,None,True,...,robust,None,trec,3.0,2.0,3.0,../data/processed/topics,topics,NaN,NaN
4,2025-11-12_22:53:47,0.83 ± 0.01,0.67 ± 0.03,0.15 ± 0.01,2025-11-12_22:53:47,qwen3-30B-no-think,robust,-DNA-zero-shot,None,True,...,robust,None,trec,5.0,3.0,3.0,../data/processed/topics,topics,NaN,NaN
5,2025-11-12_22:59:42,0.76 ± 0.01,0.53 ± 0.02,0.23 ± 0.01,2025-11-12_22:59:42,qwen3-30B-no-think,robust,-DNA-zero-shot,None,True,...,robust,None,trec-query,1.0,3.0,3.0,../data/processed/topics,topics,NaN,NaN
6,2025-11-12_23:05:33,0.79 ± 0.01,0.59 ± 0.02,0.20 ± 0.01,2025-11-12_23:05:33,qwen3-30B-no-think,robust,-DNA-zero-shot,None,True,...,robust,None,trec-query,3.0,3.0,3.0,../data/processed/topics,topics,NaN,NaN
8,2025-11-12_23:11:31,0.80 ± 0.01,0.61 ± 0.03,0.18 ± 0.01,2025-11-12_23:11:31,qwen3-30B-no-think,robust,-DNA-zero-shot,None,True,...,robust,None,trec-query,5.0,3.0,3.0,../data/processed/topics,topics,NaN,NaN
9,2025-11-12_23:17:23,0.78 ± 0.01,0.57 ± 0.02,0.21 ± 0.01,2025-11-12_23:17:23,qwen3-30B-no-think,robust,-DNA-zero-shot,None,True,...,robust,None,trec,1.0,1.0,3.0,../data/processed/topics,topics,NaN,NaN
10,2025-11-12_23:23:16,0.84 ± 0.01,0.69 ± 0.03,0.14 ± 0.01,2025-11-12_23:23:16,qwen3-30B-no-think,robust,-DNA-zero-shot,None,True,...,robust,None,trec,3.0,2.0,3.0,../data/processed/topics,topics,NaN,NaN
12,2025-11-12_23:29:06,0.87 ± 0.01,0.74 ± 0.02,0.11 ± 0.01,2025-11-12_23:29:06,qwen3-30B-no-think,robust,-DNA-zero-shot,None,True,...,robust,None,trec,5.0,3.0,3.0,../data/processed/topics,topics,NaN,NaN
